In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [1]:
import math
import numpy as np
import asyncio
import xmlschema
import networkx as nx
import polars as pl
from itertools import zip_longest

In [2]:
# from ggblab import GeoGebra
# ggb = await GeoGebra().init(use_vscode=False)

In [1]:
import ggblab

In [2]:
# ggb

In [3]:
# from ggblab.ipymagic import unregister_ggb_magic, register_ggb_magic
# unregister_ggb_magic(get_ipython())
# register_ggb_magic(get_ipython())

In [4]:
# import os
# os.environ['GGBLAB_IPYMAGIC_DEBUG'] = '1'

In [5]:
# get_ipython().user_ns['_ggb_debug']=True

In [6]:
# %reload_ext ggblab

In [7]:
# Segment(_1, _8, _10)
# Segment(_8, _9, _10)
# Segment(_9, _1, _10)

In [8]:
cmds = r'''(0, 0)
Circle(_1, 1)
Point(_2)
Line(_1, _3)
Point(_4)
PerpendicularLine(_5, _4)
{Intersect(_2, _6)}
_7(1)
_7(2)
Polygon(_1, _8, _9)
# Segment(_1, _8, _10)
# Segment(_8, _9, _10)
# Segment(_9, _1, _10)
Midpoint(_1, _5)
Circle(_, _5)
{Intersect(_, _2)}
_(1)
__(2)
Polygon(_1, _5, __)
{Tangent(_5, _2)}
'''

# for line_num, line in enumerate(cmds.splitlines(), start=1):
#     print(f"{line_num}: {line.strip()}")
cmds

'(0, 0)\nCircle(_1, 1)\nPoint(_2)\nLine(_1, _3)\nPoint(_4)\nPerpendicularLine(_5, _4)\n{Intersect(_2, _6)}\n_7(1)\n_7(2)\nPolygon(_1, _8, _9)\n# Segment(_1, _8, _10)\n# Segment(_8, _9, _10)\n# Segment(_9, _1, _10)\nMidpoint(_1, _5)\nCircle(_, _5)\n{Intersect(_, _2)}\n_(1)\n__(2)\nPolygon(_1, _5, __)\n{Tangent(_5, _2)}\n'

In [9]:
%ggb ggb {cmds}

ggblab: GeoGebra singleton created and assigned to name 'ggb' in the user namespace.


In [10]:
%ggblab api newConstruction()

In [11]:
%%ggb
(0, 0)
Circle(_1, 1)
Point(_2)
Line(_1, _3)
Point(_4)
PerpendicularLine(_5, _4)
{Intersect(_2, _6)}
_7(1)
_7(2)
Polygon(_1, _8, _9)
# Segment(_1, _8, _10)
# Segment(_8, _9, _10)
# Segment(_9, _1, _10)
Midpoint(_1, _5)
Circle(_, _5)
{Intersect(_, _2)}
_(1)
__(2)
Polygon(_1, _5, __)
{Tangent(_5, _2)}

In [9]:
_

['A', 'c', 'B', 'f', 'C', 'g', 'l1', 'D', 'E', 't1']

In [10]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [11]:
df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
1,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,0,false,false,false
2,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,true,false
3,"""B""","""point""","""Point(c)""","""B = (1, 0)""",null,0,true,true,false
4,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,true,true,false
5,"""C""","""point""","""Point(f)""","""C = (0, 0)""",null,0,true,true,false
6,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 0""",null,0,true,true,false
7,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(0, -1), (0, 1)}""",null,0,true,true,false
8,"""D""","""point""","""l1(1)""","""D = (0, -1)""",null,0,true,true,false
9,"""E""","""point""","""l1(2)""","""E = (0, 1)""",null,0,true,true,false


In [92]:
cmds = ConstructionIO.commands_for_magic(df, use_name_equals=False)
print(cmds)

(0, 0)
Circle(_1, 1)
Point(_2)
Line(_1, _3)
Point(_4)
PerpendicularLine(_5, _4)
{Intersect(_2, _6)}
_7(1)
_7(2)
Polygon(_1, _8, _9)
Segment(_1, _8, _10)
Segment(_8, _9, _10)
Segment(_9, _1, _10)


In [93]:
for line_num, line in enumerate(cmds.splitlines(), start=1):
    print(f"{line_num}: {line.strip()}")

1: (0, 0)
2: Circle(_1, 1)
3: Point(_2)
4: Line(_1, _3)
5: Point(_4)
6: PerpendicularLine(_5, _4)
7: {Intersect(_2, _6)}
8: _7(1)
9: _7(2)
10: Polygon(_1, _8, _9)
11: Segment(_1, _8, _10)
12: Segment(_8, _9, _10)
13: Segment(_9, _1, _10)


In [94]:
%ggblab api newConstruction()

In [95]:
%ggblab {cmds}

In [12]:
r1 = await ggb.function('getValueString', ['l1'])
# t1 = ggb.parser.tokenize_with_commas(r1)
t1 = ggb.parser.tokenize(r1)
t1, len(t1[2])

(['l1', '=', [['0', '-1'], ['0', '1']]], 2)

In [65]:
_oh

{9: ['A', 'c', 'B', 'f', 'C', 'g', 'l1', 'D', 'E', 't1'],
 11: shape: (13, 10)
 ┌──────────┬──────┬──────────┬────────────────────┬───┬───────┬────────────┬───────────┬───────────┐
 │ Sequence ┆ Name ┆ Type     ┆ Command            ┆ … ┆ Layer ┆ ShowObject ┆ ShowLabel ┆ Auxiliary │
 │ ---      ┆ ---  ┆ ---      ┆ ---                ┆   ┆ ---   ┆ ---        ┆ ---       ┆ ---       │
 │ u32      ┆ str  ┆ str      ┆ str                ┆   ┆ u32   ┆ bool       ┆ bool      ┆ bool      │
 ╞══════════╪══════╪══════════╪════════════════════╪═══╪═══════╪════════════╪═══════════╪═══════════╡
 │ 1        ┆ A    ┆ point    ┆ (0, 0)             ┆ … ┆ 0     ┆ false      ┆ false     ┆ false     │
 │ 2        ┆ c    ┆ circle   ┆ Circle(A, 1)       ┆ … ┆ 0     ┆ true       ┆ true      ┆ false     │
 │ 3        ┆ B    ┆ point    ┆ Point(c)           ┆ … ┆ 0     ┆ true       ┆ true      ┆ false     │
 │ 4        ┆ f    ┆ line     ┆ Line(A, B)         ┆ … ┆ 0     ┆ true       ┆ true      ┆ false     │
 │ 

In [4]:
%pwd

'/Users/manabu/work/ggblab/examples'

In [7]:
%cd ggblab/examples

/Users/manabu/work/ggblab/examples


In [4]:
ggb.file.load('eg11_slider.ggb')
# ggb.file.load('2025_06_08.ggb')

In [5]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [51]:
from ggblab.errors import GeoGebraError, GeoGebraSyntaxError, GeoGebraSemanticsError, GeoGebraAppletError

In [56]:
ggb.check_semantics = False

In [13]:
await ggb.command('(0, 0)')

'A'

In [14]:
await ggb.command('Circle(A, 1)')

'c'

In [15]:
await ggb.command('Point(c)')

'B'

In [7]:
await ggb.command('Line(A, B)')

'f'

In [8]:
await ggb.command('Point(f)')

'C'

In [9]:
await ggb.command('PerpendicularLine(C, f)')

'g'

In [10]:
await ggb.command('{Intersect(c, g)}')

'l1'

In [11]:
await ggb.command('l1(1)')

'D'

In [12]:
await ggb.command('l1(2)')

'E'

In [13]:
await ggb.command('t1 = Polygon(A, D, E)')

't1,e,a,d'

In [35]:
p.df.write_parquet('df_ref')

In [42]:
p_ref.df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary,DependsOn,DependsOn_minimal
u32,str,str,str,str,str,u32,bool,bool,bool,list[str],list[str]
0,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,0,false,false,false,[],[]
1,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,true,false,"[""A""]","[""A""]"
2,"""B""","""point""","""Point(c)""","""B = (-1, 0)""",null,0,false,false,false,"[""A"", ""c""]","[""c""]"
3,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,false,false,false,"[""A"", ""B"", ""c""]","[""c""]"
4,"""C""","""point""","""Point(f)""","""C = (1, 0)""",null,0,false,false,false,"[""A"", ""B"", … ""f""]","[""B"", ""f""]"
5,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 1""",null,0,false,false,false,"[""A"", ""B"", … ""f""]","[""B"", ""f""]"
6,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(1, 0)}""",null,0,true,true,false,[],[]
7,"""D""","""point""","""l1(1)""","""D = (1, 0)""",null,0,false,false,false,"[""l1""]","[""l1""]"
8,"""E""","""point""","""l1(2)""","""E = (?, ?)""",null,0,true,true,false,"[""l1""]","[""l1""]"


In [36]:
p_ref = ConstructionTreeParser(p.df)

In [39]:
g_ref = p_ref.parse()
labels_map = {}
for n, t in p.df["Name", "Type"].rows():
    try:
        g_ref.nodes[n]['label'] = f"{n} ({t})"
    except:
        pass

In [40]:
nx.set_node_attributes(g1, labels_map, "label")
nx.write_network_text(g1, with_labels="label")

╟── A (point)
╎   ├─╼ c (circle)
╎   │   └─╼ B (point)
╎   │       └─╼ f (line) ╾ A (point)
╎   │           ├─╼ C (point)
╎   │           │   └─╼ g (line) ╾ f (line)
╎   │           └─╼  ...
╎   ├─╼ t1 (triangle) ╾ D (point), E (point)
╎   │   ├─╼ h (segment) ╾ A (point), D (point)
╎   │   ├─╼ i (segment) ╾ D (point), E (point)
╎   │   └─╼ j (segment) ╾ E (point), A (point)
╎   └─╼  ...
╙── l1 (list)
    ├─╼ D (point)
    │   └─╼  ...
    └─╼ E (point)
        └─╼  ...


In [11]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [12]:
df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
1,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,0,false,false,false
2,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,true,false
3,"""B""","""point""","""Point(c)""","""B = (1, 0)""",null,0,true,true,false
4,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,true,true,false
5,"""C""","""point""","""Point(f)""","""C = (0, 0)""",null,0,true,true,false
6,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 0""",null,0,true,true,false
7,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(0, -1), (0, 1)}""",null,0,true,true,false
8,"""D""","""point""","""l1(1)""","""D = (0, -1)""",null,0,true,true,false
9,"""E""","""point""","""l1(2)""","""E = (0, 1)""",null,0,true,true,false


In [13]:
cmds = ConstructionIO.commands_for_magic(df)
cmds

'(0, 0)\nCircle(_1, 1)\nPoint(_2)\nLine(_1, _3)\nPoint(_4)\nPerpendicularLine(_5, _4)\n{Intersect(_2, _6)}\n_7(1)\n_7(2)\nPolygon(_1, _8, _9)\nSegment(_1, _8, _10)\nSegment(_8, _9, _10)\nSegment(_9, _1, _10)'

In [15]:
p = ConstructionTreeParser(df)
g1 = p.parse()

In [16]:
labels_map = {}
for n, t in p.df["Name", "Type"].rows():
    try:
        g1.nodes[n]['label'] = f"{n} ({t})"
    except:
        pass

In [17]:
nx.set_node_attributes(g1, labels_map, "label")
nx.write_network_text(g1, with_labels="label")

╟── A (point)
╎   ├─╼ c (circle)
╎   │   └─╼ B (point)
╎   │       └─╼ f (line) ╾ A (point)
╎   │           ├─╼ C (point)
╎   │           │   └─╼ g (line) ╾ f (line)
╎   │           └─╼  ...
╎   ├─╼ t1 (triangle) ╾ D (point), E (point)
╎   │   ├─╼ e (segment) ╾ A (point), D (point)
╎   │   ├─╼ a (segment) ╾ D (point), E (point)
╎   │   └─╼ d (segment) ╾ E (point), A (point)
╎   └─╼  ...
╙── l1 (list)
    ├─╼ D (point)
    │   └─╼  ...
    └─╼ E (point)
        └─╼  ...


In [18]:
g2 = p.parse_subgraph()
nx.write_network_text(g2)

╙── A
    └─╼ c
        ├─╼ B
        │   ├─╼ g ╾ f
        │   └─╼ C ╾ f
        └─╼ f
            └─╼  ...


In [19]:
from ggblab_extra import hungarian_similarity

In [21]:
s, r = hungarian_similarity(g1, g2)
s

np.float64(0.675)

In [33]:
p.df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary,DependsOn,DependsOn_minimal
u32,str,str,str,str,str,u32,bool,bool,bool,list[str],list[str]
0,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,0,false,false,false,[],[]
1,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,true,false,"[""A""]","[""A""]"
2,"""B""","""point""","""Point(c)""","""B = (-1, 0)""",null,0,false,false,false,"[""A"", ""c""]","[""c""]"
3,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,false,false,false,"[""A"", ""B"", ""c""]","[""c""]"
4,"""C""","""point""","""Point(f)""","""C = (1, 0)""",null,0,false,false,false,"[""A"", ""B"", … ""f""]","[""B"", ""f""]"
5,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 1""",null,0,false,false,false,"[""A"", ""B"", … ""f""]","[""B"", ""f""]"
6,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(1, 0)}""",null,0,true,true,false,[],[]
7,"""D""","""point""","""l1(1)""","""D = (1, 0)""",null,0,false,false,false,"[""l1""]","[""l1""]"
8,"""E""","""point""","""l1(2)""","""E = (?, ?)""",null,0,true,true,false,"[""l1""]","[""l1""]"


In [12]:
l1 = 'n'
l2 = 'i'

In [13]:
await ggb.listen(l1, True)
await ggb.listen(l2, True)

{}

In [141]:
await ggb.listen('a', False)
await ggb.listen('b', False)

{}

In [14]:
ggb.comm.shared_objects

{'n': 'n = 0', 'i': 'i = 0'}

In [15]:
# import ipywidgets as widgets
# label1 = widgets.Label(value=ggb.comm.shared_objects['n'])
# label2 = widgets.Label(value=ggb.comm.shared_objects['m'])
# display(label1, label2)

In [16]:
async def on_shared_update1(changes):
    # await asyncio.sleep(0)
    # label1.value = changes['a']
    n = int(changes[l1].split()[2])
    # await asyncio.sleep(0)
    await ggb.function("setLayerVisible", list(zip_longest(range(9), [True]*n, fillvalue=False)))
    r = await ggb.function('getXML', [l2])
    o = ggb.file.ggb_schema.decode(r)
    o['value'][0]['@val'] = '0'
    x = xmlschema.etree_tostring(ggb.file.ggb_schema.encode(o, 'element'))
    r = await ggb.function('evalXML' , [x])

In [17]:
ggb.comm.remove_shared_listener(on_shared_update1)
ggb.comm.add_shared_listener(on_shared_update1)

True

In [18]:
async def on_shared_update2(changes):
    # await asyncio.sleep(0)
    # label2.value = changes['b']
    m = int(changes[l2].split()[2])
    n = int(ggb.comm.shared_objects[l1].split()[2])
    l = df.filter(pl.col("Layer") == n)["Name"].to_list()
    # m = int(ggb.comm.shared_objects['m'].split()[2])
    # list(zip_longest(l, [True]*m, fillvalue=False))
    await ggb.function("setVisible", list(zip_longest(l, [True]*m, fillvalue=False)))

In [19]:
ggb.comm.remove_shared_listener(on_shared_update2)
ggb.comm.add_shared_listener(on_shared_update2)

True

In [104]:
ggb.comm.clear_shared_listeners()

2

In [29]:
await ggb.command('Curve(x, x^2, x, -10, 10)')

'a'

In [30]:
await ggb.command('a(b)')

'G'

In [32]:
await ggb.command('Tangent(G, a)')

'h'

In [33]:
await ggb.command('u w')

'e'

In [34]:
await ggb.command('sqrt(u u)')

'j'

In [36]:
await ggb.command('k = Distance(a,F)')

'k'

In [7]:
await ggb.command('{Tangent(c,C)}')

'l2'

In [17]:
r1 = await ggb.function('getValueString', ['l1'])
# l1 = ggb.parser.tokenize_with_commas(r1)
l2 = ggb.parser.tokenize(r1)
l2

['l1',
 '=',
 [['0.6052896362051', '-0.7960053117302'],
  ['0.6052896362051', '0.7960053117302']]]

In [18]:
r2 = await ggb.function('getValueString', ['l2'])
l = ggb.parser.tokenize(r2, simplify=True)
l

[]

In [52]:
await ggb.command("l1(2)")

'E'

In [50]:
await ggb.command("{Intersect[c, g]}")

'l1'

In [200]:
async def getCoords(list_points=[]):
    r = await ggb.function(["getXcoord", "getYcoord"], [[p] for p in list_points])
    arr = np.array(list(zip(*r)), dtype=float)
    arr[np.isclose(arr, 0., atol=1e-9)] = 0.
    arr = arr[~np.any(np.isnan(arr), axis=1)]
    return arr.tolist()

In [219]:
await getCoords(['D', 'E'])

[[0.8000000000000002, 0.5999999999999999], [0.7999999999999999, -0.6]]

In [222]:
r = await ggb.function("getValueString", ['l1'])
len(list(toCoords(r)))

2

In [215]:
from collections.abc import Iterable

def toCoords(r):
    # ret = []
    for e in ggb.parser.tokenize_with_commas(r):
        if isinstance(e, Iterable) and not isinstance(e, (str, bytes)):
            r2 = [float(e2) for e2 in e if e2 not in [',', '?']]
            if r2:
                # ret.append(r2)
                yield r2
    # return ret


In [223]:
import ipywidgets as widgets
label = widgets.Label(value="")
display(label)

Label(value='')

In [224]:
await ggb.listen('l1')

{}

In [225]:
ggb.comm.shared_objects

{'l1': 'l1 = {(0.8, 0.6), (0.8, -0.6)}'}

In [226]:
async def on_shared_update(changes):
    n = len(list(toCoords(changes['l1'])))
    label.value = f"Length of l1: {n}"

In [227]:
ggb.comm.add_shared_listener(on_shared_update)

True

In [6]:
%pwd

'/Users/manabu/work/ggblab'

In [ ]:
# ggb.file.source_file = 'eg11_slider.ggb'

In [39]:
ggb.file.base64_buffer = await ggb.function("getBase64")

In [40]:
ggb.file.save(overwrite=True)

* 原則（教育観）:
    - 目的化: 再現は「結果」ではなく「理解（なぜその操作か）」を目的にする。
    - 予測→検証: 次に何が起きるか予測させてから操作させる。
    - 説明要求: 手順ごとに短い理由説明（1文）を書かせる。
    - 変奏課題: パラメータを少し変えた課題で本質が移るか確認する。
    - 生成的課題: 「同じ発想で別の図形を作る」など転移を問う。
* ggblabで実装できる仕組み（短）:
    - 段階公開（layer slider）: 各レイヤーに「解説」「問い」「期待する操作」を紐付け、スライダーで段階的に提示。
    - 予測プロンプト: 各ステップの前に「次に何が起きる？」を表示し、回答を記録。
    - 説明入力欄: 学生が操作毎に短い説明を入力 → 教師や自動ルールでフィードバック。
    - 変化タスク自動化: DataFrame→コマンド生成を利用してパラメータをランダム化した派生課題を作る。
    - 操作ログ＋解析: 操作順・所要時間・試行回数をログ化して学習診断に使う。
    - 差分フィードバック: 学生構成と模範構成を比較して「次に直すべき一手」を提示。

In [29]:
await ggb.function("getVersion")

'5.2.909.9'